# 第 55 章：Capstone 应急演练与替班教师接管手册

这个 notebook 用一个极小的 substitute-handoff table，对比“只按 readiness 分数排序”的 naive baseline 和“先检查升级边界、owner、最小运行包与上下文 brief”的替班接管 workflow。

In [ ]:
from __future__ import annotations

import csv
import platform
from collections import Counter
from pathlib import Path

DATA_PATH = Path('../../data/small/capstone_contingency_substitute_handoff_demo.csv')


def load_items(path):
    rows = []
    with path.open(encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            row['readiness_score'] = float(row['readiness_score'])
            rows.append(row)
    return rows


def ordered_counts(rows, field):
    counts = Counter(row[field] for row in rows)
    return {key: counts[key] for key in sorted(counts)}


rows = load_items(DATA_PATH)
print(f'Loaded {len(rows)} substitute-handoff items from {DATA_PATH.name}')
print('Python version:', platform.python_version())
print('Trigger status:', ordered_counts(rows, 'trigger_status'))
print('Minimal run package:', ordered_counts(rows, 'minimal_run_package_status'))
print('Substitute owner:', ordered_counts(rows, 'substitute_owner_status'))
print('Context brief:', ordered_counts(rows, 'context_brief_status'))
print('Escalation status:', ordered_counts(rows, 'escalation_status'))
print('Reference route:', ordered_counts(rows, 'reference_route'))


In [ ]:
reference_ready_ids = sorted(
    row['handoff_id']
    for row in rows
    if row['reference_route'] == 'substitute_ready'
)
top4_readiness = sorted(
    rows,
    key=lambda row: row['readiness_score'],
    reverse=True,
)[:4]
baseline_ready_hits = sum(
    row['reference_route'] == 'substitute_ready'
    for row in top4_readiness
)
baseline_not_ready_hits = sum(
    row['reference_route'] != 'substitute_ready'
    for row in top4_readiness
)
baseline_ready_precision = baseline_ready_hits / len(top4_readiness)
baseline_not_ready_intrusion = baseline_not_ready_hits / len(top4_readiness)

print('Reference substitute-ready items:', reference_ready_ids)
print('Top-4 by readiness:')
for row in top4_readiness:
    print(
        f"  {row['handoff_id']}: readiness={row['readiness_score']:.2f}, "
        f"route={row['reference_route']}, area={row['contingency_area']}"
    )
print('Baseline substitute precision:', round(baseline_ready_precision, 3))
print('Baseline not-ready intrusion:', round(baseline_not_ready_intrusion, 3))


In [ ]:
def route_substitute_item(row):
    if row['escalation_status'] in {'policy', 'privacy', 'grading'}:
        return 'escalate_before_substitution', 'scenario crosses a boundary and must be escalated first'
    if row['substitute_owner_status'] != 'assigned':
        return 'assign_substitute_owner', 'no substitute owner is assigned yet'
    if row['minimal_run_package_status'] != 'ready':
        return 'complete_minimal_run_package', 'minimal run package is not complete enough'
    if row['context_brief_status'] != 'ready':
        return 'refresh_context_brief', 'substitute context brief is stale or incomplete'
    return 'substitute_ready', 'substitute can take over with a bounded, ready minimal run package'


routed_rows = []
for row in rows:
    route, reason = route_substitute_item(row)
    routed = dict(row)
    routed['route'] = route
    routed['reason'] = reason
    routed_rows.append(routed)

print('Substitute-handoff workflow routes:')
for row in routed_rows:
    print(
        f"  {row['handoff_id']}: route={row['route']}, "
        f"reference={row['reference_route']}, reason={row['reason']}"
    )


In [ ]:
ready_queue = [row for row in routed_rows if row['route'] == 'substitute_ready']
package_queue = [row for row in routed_rows if row['route'] == 'complete_minimal_run_package']
owner_queue = [row for row in routed_rows if row['route'] == 'assign_substitute_owner']
brief_queue = [row for row in routed_rows if row['route'] == 'refresh_context_brief']
escalation_queue = [row for row in routed_rows if row['route'] == 'escalate_before_substitution']

workflow_accuracy = sum(
    row['route'] == row['reference_route']
    for row in routed_rows
) / len(routed_rows)
ready_precision = sum(
    row['reference_route'] == 'substitute_ready'
    for row in ready_queue
) / max(len(ready_queue), 1)

print('Substitute-ready queue:', [row['handoff_id'] for row in ready_queue])
print('Complete-package queue:', [row['handoff_id'] for row in package_queue])
print('Assign-owner queue:', [row['handoff_id'] for row in owner_queue])
print('Refresh-brief queue:', [row['handoff_id'] for row in brief_queue])
print('Escalate-first queue:', [row['handoff_id'] for row in escalation_queue])
print('Workflow route accuracy:', round(workflow_accuracy, 3))
print('Structured substitute precision:', round(ready_precision, 3))


In [ ]:
def substitute_steps(row):
    steps = []
    if row['escalation_status'] in {'policy', 'privacy', 'grading'}:
        steps.append('先升级到主讲教师或课程负责人，不要让替班人独自处理边界问题。')
    if row['substitute_owner_status'] != 'assigned':
        steps.append('明确替班 owner，并记录课堂前后谁接手。')
    if row['minimal_run_package_status'] != 'ready':
        steps.append('补齐最小运行包：今天最少目标、入口链接、slides、notebook 和签到/提交口。')
    if row['context_brief_status'] != 'ready':
        steps.append('刷新上下文 brief：今天节奏、危险边界、课堂 caveat 和事后交回记录。')
    return steps or ['可以交给替班教师或替班助教接手；保留接管记录和课后交回 note。']


for row in routed_rows:
    if row['route'] != 'substitute_ready':
        print(f"\n{row['handoff_id']} -> {row['route']} ({row['contingency_area']})")
        for step in substitute_steps(row):
            print(' -', step)


In [ ]:
substitute_outline = [
    'Minimal goal: what this class must accomplish even under substitute conditions',
    'Entry pack: handout, notebook, slides, rubric, and submission link',
    'Boundary note: what substitute staff may decide locally and what must be escalated',
    'Timing note: the shortest workable classroom timeline',
    'Contact ladder: who answers content, policy, grading, or technical questions',
    'Return note: what happened, what changed, and what the main instructor needs to know',
]

contingency_assistant_prompt = '''你是我的 capstone 替班接管助手。

任务：
1. 先阅读 substitute-handoff table，不要只按 readiness 排序；
2. 先检查 escalation status；
3. 再检查 substitute owner、minimal run package 和 context brief；
4. 把每个场景 route 到 substitute_ready、complete_minimal_run_package、
   assign_substitute_owner、refresh_context_brief 或 escalate_before_substitution；
5. 对每个非 ready 场景输出替班前 checklist。

输出格式：
- Substitute-ready scenarios
- Complete-package scenarios
- Assign-owner scenarios
- Refresh-brief scenarios
- Escalate-first scenarios
- Substitute checklist by scenario
'''

print('Substitute handoff outline:')
for item in substitute_outline:
    print(' -', item)

print('\nAssistant prompt:')
print(contingency_assistant_prompt)


In [ ]:
substitute_snapshot = {
    'dataset': DATA_PATH.name,
    'n_handoffs': len(rows),
    'baseline_top4_substitute_precision': round(baseline_ready_precision, 3),
    'baseline_not_ready_intrusion': round(baseline_not_ready_intrusion, 3),
    'workflow_route_accuracy': round(workflow_accuracy, 3),
    'substitute_ready': [row['handoff_id'] for row in ready_queue],
    'complete_minimal_run_package': [row['handoff_id'] for row in package_queue],
    'assign_substitute_owner': [row['handoff_id'] for row in owner_queue],
    'refresh_context_brief': [row['handoff_id'] for row in brief_queue],
    'escalate_before_substitution': [row['handoff_id'] for row in escalation_queue],
    'python_version': platform.python_version(),
}

print('Substitute handoff snapshot:')
for key, value in substitute_snapshot.items():
    print(f'  {key}: {value}')
